# Milestone 2 Cleaning Script - Copy for Sampling

- This script is copied from Milestone 1 to generate a sample of 5000 cases for Milestone 2

In [ ]:
import pandas as pd
import pm4py
import numpy as np
import os
from openai import OpenAI

In [3]:
df_noised = pm4py.read_xes(
    "../data/samples/noised_sample.xes.gz",
    return_legacy_log_object=False
)

print(df_noised.shape)

c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 5000/5000 [00:10<00:00, 472.95it/s]


(151673, 21)


## Clean unanchored Events / timestamp column
Coded by: Felix

In [5]:
df_unanchored = pd.read_csv("../data/samples/unanchored_events_sample.csv.gz", sep=";")

df_unanchored.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount
0,Created,User_1,A_Create Application,Application,Application_794921244,complete,24.11.2016 14:59:29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remaining debt home,New credit,Application_794921244,25500.0
1,statechange,User_1,A_Submitted,Application,ApplState_807426416,complete,24.11.2016 14:59:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remaining debt home,New credit,Application_794921244,25500.0
2,Created,User_1,W_Handle leads,Workflow,Workitem_673122953,schedule,2016-11-24 14:59:30.868000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remaining debt home,New credit,Application_794921244,25500.0
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1420938908,withdraw,2016-11-24 15:00:30.540000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remaining debt home,New credit,Application_794921244,25500.0
4,Created,User_1,W_Complete application,Workflow,Workitem_1612441895,schedule,2016-11-24 15:00:30.547000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remaining debt home,New credit,Application_794921244,25500.0


In [6]:
# identify different time formats
s = df_unanchored["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 152388
European dd.mm.yyyy 37867
European d.mm.yyyy 37867
US-like mm/dd/yyyy 0
invalid / custom strings 0
UTC 30926


C:\Users\thien\AppData\Local\Temp\ipykernel_4444\2464342000.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


In [7]:
def clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
):
    df = df.copy()

    timestamps = df[timestamp_column].astype("string")

    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

    # Format 1: German format, e.g. 31.12.2017 14:30:00
    mask_german = timestamps.str.match(
        r"^\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}:\d{2}$",
        na=False
    )

    parsed.loc[mask_german] = pd.to_datetime(
        timestamps.loc[mask_german],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
        utc=True
    )

    # Format 2: ISO with T and Z, e.g. 2017-12-31T14:30:00.123456Z
    mask_iso_z = timestamps.str.match(
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$",
        na=False
    )

    parsed.loc[mask_iso_z] = pd.to_datetime(
        timestamps.loc[mask_iso_z],
        format="%Y-%m-%dT%H:%M:%S.%fZ",
        errors="coerce",
        utc=True
    )

    # Fallback: try normal pandas parsing for unchanged/default timestamps
    mask_remaining = parsed.isna() & timestamps.notna()

    parsed.loc[mask_remaining] = pd.to_datetime(
        timestamps.loc[mask_remaining],
        errors="coerce",
        utc=True
    )

    df[timestamp_column] = parsed

    return df

In [8]:
df_cleaned_unanchored = clean_unanchored_events_mixed_formats(
    df_unanchored,
    timestamp_column="time:timestamp"
)

In [9]:
s = df_cleaned_unanchored["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 190136
European dd.mm.yyyy 0
European d.mm.yyyy 0
US-like mm/dd/yyyy 0
invalid / custom strings 0
UTC 0


C:\Users\thien\AppData\Local\Temp\ipykernel_4444\1314539912.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


In [10]:
# check for null values in timestamp column
df_check = df_cleaned_unanchored.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application     44
Offer           35
Workflow       119
Name: time:timestamp, dtype: int64


In [11]:
application_na_rows = df_check[
    (df_check["EventOrigin"] == "Application") &
    (df_check["time:timestamp"].isna())
]
application_na_rows.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount
4499,statechange,User_80,A_Complete,Application,ApplState_242795721,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_393314030,12500.0
9314,Created,User_1,A_Create Application,Application,Application_96732613,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home improvement,New credit,Application_96732613,7500.0
15000,statechange,User_30,A_Incomplete,Application,ApplState_2009367802,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_1288650428,15000.0
22265,statechange,User_100,A_Pending,Application,ApplState_1179576121,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Car,New credit,Application_1485367187,7000.0
24019,statechange,User_117,A_Validating,Application,ApplState_714511529,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home improvement,New credit,Application_140289375,30000.0


In [12]:
# check whole trace of one event with missing timestamp
case_id = application_na_rows.iloc[1]["case:concept:name"]

df_check.loc[
    df_check["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "EventID"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,EventID
9314,Application_96732613,A_Create Application,Application,NaT,Application_96732613
9315,Application_96732613,A_Submitted,Application,2016-10-23 18:03:18+00:00,ApplState_558618280
9316,Application_96732613,W_Handle leads,Workflow,2016-10-23 18:03:19.057000+00:00,Workitem_594136729
9317,Application_96732613,W_Handle leads,Workflow,2016-10-23 18:04:23.441000+00:00,Workitem_1381539282
9318,Application_96732613,W_Complete application,Workflow,2016-10-23 18:04:23.448000+00:00,Workitem_163997577
9319,Application_96732613,A_Concept,Application,2016-10-23 18:04:23+00:00,ApplState_840889758
9320,Application_96732613,A_Accepted,Application,2016-10-26 12:46:09+00:00,ApplState_2017635843
9321,Application_96732613,O_Create Offer,Offer,2016-10-26 12:50:49.623000+00:00,Offer_1759354440
9322,Application_96732613,O_Created,Offer,2016-10-26 12:50:50.172000+00:00,OfferState_22019692
9323,Application_96732613,O_Create Offer,Offer,2016-10-26 12:53:58.309000+00:00,Offer_2117302687


In [13]:
df_check["event_order"] = range(len(df_check))

In [14]:
def impute_missing_timestamps_within_cases(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()
    imputable_mask = missing_mask & previous_time.notna() & next_time.notna()

    df.loc[imputable_mask, timestamp_column] = (
        previous_time[imputable_mask]
        + (next_time[imputable_mask] - previous_time[imputable_mask]) / 2
    )

    df.loc[imputable_mask, "timestamp_cleaning"] = "imputed_between_neighbors"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [15]:
df_cleaned_unanchored = impute_missing_timestamps_within_cases(
    df_check,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column="event_order"
)

In [16]:
df_cleaned_unanchored.loc[
    df_cleaned_unanchored["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "timestamp_cleaning"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,timestamp_cleaning
9314,Application_96732613,A_Create Application,Application,NaT,NaN
9315,Application_96732613,A_Submitted,Application,2016-10-23 18:03:18+00:00,NaN
9316,Application_96732613,W_Handle leads,Workflow,2016-10-23 18:03:19.057000+00:00,NaN
9317,Application_96732613,W_Handle leads,Workflow,2016-10-23 18:04:23.441000+00:00,NaN
9318,Application_96732613,W_Complete application,Workflow,2016-10-23 18:04:23.448000+00:00,NaN
9319,Application_96732613,A_Concept,Application,2016-10-23 18:04:23+00:00,NaN
9320,Application_96732613,A_Accepted,Application,2016-10-26 12:46:09+00:00,NaN
9321,Application_96732613,O_Create Offer,Offer,2016-10-26 12:50:49.623000+00:00,NaN
9322,Application_96732613,O_Created,Offer,2016-10-26 12:50:50.172000+00:00,NaN
9323,Application_96732613,O_Create Offer,Offer,2016-10-26 12:53:58.309000+00:00,NaN


In [17]:
# check for null values in timestamp column
df_check = df_cleaned_unanchored.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    8
Offer          0
Workflow       1
Name: time:timestamp, dtype: int64


In [18]:
def fill_missing_timestamps_with_neighbor(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None,
    cleaning_column="timestamp_cleaning"
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()

    previous_mask = missing_mask & previous_time.notna()
    df.loc[previous_mask, timestamp_column] = previous_time[previous_mask]
    df.loc[previous_mask, cleaning_column] = "filled_from_previous_timestamp"

    missing_mask = df[timestamp_column].isna()

    next_mask = missing_mask & next_time.notna()
    df.loc[next_mask, timestamp_column] = next_time[next_mask]
    df.loc[next_mask, cleaning_column] = "filled_from_next_timestamp"

    missing_mask = df[timestamp_column].isna()
    df.loc[missing_mask, cleaning_column] = "could_not_fill_timestamp"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [19]:
df_cleaned_unanchored = fill_missing_timestamps_with_neighbor(
    df_cleaned_unanchored,
    case_column="case:concept:name",
    timestamp_column="time:timestamp"
)

In [20]:
df_cleaned_unanchored["timestamp_cleaning"].value_counts()

timestamp_cleaning
imputed_between_neighbors         189
filled_from_next_timestamp          8
filled_from_previous_timestamp      1
Name: count, dtype: int64

In [21]:
#export cleaned unanchored events to csv
df_cleaned_unanchored.to_csv("../data/samples/cleaned_unanchored_events_sample.csv.gz", index=False, sep=";", compression="gzip")

## Cleaning Other Patterns

In [22]:
df_cleaned = df_noised.merge(
    df_cleaned_unanchored[["EventID", "time:timestamp"]],
    on="EventID",
    how="left",
    suffixes=("_noised", "_clean")
)

df_cleaned["time:timestamp"] = df_cleaned["time:timestamp_clean"]

df_cleaned = df_cleaned.drop(columns=["time:timestamp_noised", "time:timestamp_clean"])

df_cleaned.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,...,CreditScore,OfferedAmount,OfferID,pollution_type,start_timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,time:timestamp
0,Created,User_1,A_Create App.,Application,Application_100034150,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,distorted_label,NaT,Existing loan takeover,New credit,Application_100034150,5000.0,2016-02-26 08:17:08+00:00
1,statechange,User_1,A_Submitted,Application,ApplState_1876536887,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,Existing loan takeover,New credit,Application_100034150,5000.0,2016-02-26 08:17:08+00:00
2,Created,User_1,W_Handle leads,Workflow,Workitem_1857743533,schedule,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,Existing loan takeover,New credit,Application_100034150,5000.0,2016-02-26 08:17:09.360000+00:00
3,Obtained,User_5,W_Handle leads,Workflow,Workitem_1381927478,start,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,Existing loan takeover,New credit,Application_100034150,5000.0,2016-02-26 09:18:19.304000+00:00
4,Deleted,User_5,W_Handle leads,Workflow,Workitem_1723165043,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,Existing loan takeover,New credit,Application_100034150,5000.0,2016-02-26 09:18:43.550000+00:00


### Clean Polluted Labels
Coded by: Felix

In [23]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                                             28849
W_Call incomplete files                                         25459
W_Complete application                                          15637
W_Handle leads                                                   7169
O_Create Offer                                                   6533
                                                                ...  
W_Complete application - Incident No. Application_909561500         1
A_Create Application - Incident No. Application_91048792            1
W_Handle leads - Incident No. Application_925907733                 1
O_Sent (mail and online) - Incident No. Application_96304459        1
W_Call after offers - Incident No. Application_981969053            1
Name: count, Length: 202, dtype: int64

In [24]:
def clean_polluted_labels(
    df,
    activity_column="concept:name",
    cleaning_column="label_cleaning"
):
    df = df.copy()

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    original_labels = df[activity_column].copy()

    # convert as string
    df[activity_column] = df[activity_column].astype("string")

    patterns = [
        r"\._\d+$",                              # ._1691306052
        r"_\d+$",                                # _1691306052
        r"\s*-\s*Incident No\.?\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Incident\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Case\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Application\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*User\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Resource\s*[A-Za-z0-9_\-]+",
        r"\s*#\s*[A-Za-z0-9_\-]+",
        r"\s*\(\s*id\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
        r"\s*\(\s*case\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
    ]

    for pattern in patterns:
        df[activity_column] = df[activity_column].str.replace(
            pattern,
            "",
            regex=True
        )

    # normalize whitespace
    df[activity_column] = (
        df[activity_column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    changed_mask = original_labels.astype("string") != df[activity_column]

    df.loc[changed_mask, cleaning_column] = "cleaned_polluted_label"

    return df

In [25]:
df_cleaned = clean_polluted_labels(
    df_cleaned,
    activity_column="concept:name"
)

In [26]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                           28876
W_Call incomplete files                       25493
W_Complete application                        15657
W_Handle leads                                 7176
O_Create Offer                                 6538
O_Created                                      6515
O_Sent (mail and online)                       6049
A_Complete                                     4757
A_Create Application                           4757
A_Accepted                                     4731
A_Concept                                      4722
O_Returned                                     3525
A_Incomplete                                   3407
W_Finalize application                         3399
W_Finish application                           3302
O_Cancelled                                    3225
A_Submitted                                    3083
O_Accepted                                     2607
A_Pending                                      2594

### Clean Distorted Labels
Coded by: Felix

In [27]:
DISTORTED_LABEL_MAPPING = {
    "W": {
        "Validate appl.": "Validate application",
        "Call after ofrs.": "Call after offers",
        "Call incompl. docs.": "Call incomplete files",
        "Call incompl. files": "Call incomplete files",
        "Complete appl.": "Complete application",
        "Handle lds.": "Handle leads",
        "Assess pot. frd.": "Assess potential fraud",
        "Assess pot. fraud": "Assess potential fraud",
        "Short. compl.": "Shortened completion",
        "Short. completion": "Shortened completion",
        "Pers. Ln. coll.": "Personal Loan collection",
        "Pers. Loan collection": "Personal Loan collection",
    },
    "O": {
        "Create Off.": "Create Offer",
        "Crtd.": "Created",
        "Sent (email and onl.)": "Sent (mail and online)",
        "Sent (mail and onl.)": "Sent (mail and online)",
        "Sent (onl. only)": "Sent (online only)",
        "Ret.": "Returned",
        "Canc.": "Cancelled",
        "Acc.": "Accepted",
        "Ref.": "Refused",
    },
    "A": {
        "Valid.": "Validating",
        "Create App.": "Create Application",
        "Conc.": "Concept",
        "Concept.": "Concept",
        "Acc.": "Accepted",
        "Comp.": "Complete",
        "Incompl.": "Incomplete",
        "Subm.": "Submitted",
        "Pend.": "Pending",
        "Canc.": "Cancelled",
        "Den.": "Denied",
    }
}

In [28]:
def build_label_mapping(grouped_mapping):
    mapping = {}

    for prefix, labels in grouped_mapping.items():
        for distorted, cleaned in labels.items():
            mapping[f"{prefix}_{distorted}"] = f"{prefix}_{cleaned}"

    return mapping

In [29]:
label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

In [30]:
label_mapping

{'W_Validate appl.': 'W_Validate application',
 'W_Call after ofrs.': 'W_Call after offers',
 'W_Call incompl. docs.': 'W_Call incomplete files',
 'W_Call incompl. files': 'W_Call incomplete files',
 'W_Complete appl.': 'W_Complete application',
 'W_Handle lds.': 'W_Handle leads',
 'W_Assess pot. frd.': 'W_Assess potential fraud',
 'W_Assess pot. fraud': 'W_Assess potential fraud',
 'W_Short. compl.': 'W_Shortened completion',
 'W_Short. completion': 'W_Shortened completion',
 'W_Pers. Ln. coll.': 'W_Personal Loan collection',
 'W_Pers. Loan collection': 'W_Personal Loan collection',
 'O_Create Off.': 'O_Create Offer',
 'O_Crtd.': 'O_Created',
 'O_Sent (email and onl.)': 'O_Sent (mail and online)',
 'O_Sent (mail and onl.)': 'O_Sent (mail and online)',
 'O_Sent (onl. only)': 'O_Sent (online only)',
 'O_Ret.': 'O_Returned',
 'O_Canc.': 'O_Cancelled',
 'O_Acc.': 'O_Accepted',
 'O_Ref.': 'O_Refused',
 'A_Valid.': 'A_Validating',
 'A_Create App.': 'A_Create Application',
 'A_Conc.': 'A_Con

In [31]:
def clean_distorted_labels(
    df,
    activity_column="concept:name"
):
    df = df.copy()

    label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

    df[activity_column] = df[activity_column].replace(label_mapping)

    return df

In [32]:
df_cleaned = clean_distorted_labels(df_cleaned)

In [33]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                           30403
W_Call incomplete files                       26796
W_Complete application                        16875
W_Handle leads                                 7553
O_Create Offer                                 6869
O_Created                                      6869
O_Sent (mail and online)                       6372
A_Create Application                           5000
A_Concept                                      5000
A_Accepted                                     5000
A_Complete                                     4979
O_Returned                                     3702
A_Incomplete                                   3588
W_Finalize application                         3399
O_Cancelled                                    3388
W_Finish application                           3302
A_Submitted                                    3255
O_Accepted                                     2741
A_Pending                                      2741

### Clean Scattered Cases
Coded by: Jonas

In [35]:
# read CSV containing the scattered rows
validation_df = pd.read_csv("../data/samples/scattered_events_sample.csv.gz", sep=",")

In [36]:
# reindex
validation_df = validation_df.reindex(columns=df_cleaned.columns)

# merging the two dataframes
df_cleaned = pd.concat([df_cleaned, validation_df], ignore_index=True)

# parse timestamps
df_cleaned["time:timestamp"] = pd.to_datetime(
    df_cleaned["time:timestamp"],
    #errors="coerce",
    utc=True
)

### Clean Synonymous Labels
Coded by: Thien

In [ ]:
ENDPOINT = os.getenv("OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("OPENAI_DEPLOYMENT_NAME")
API_KEY = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY,
)

In [38]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [119]:
def find_similar_labels(
    df: pd.DataFrame,
    threshold: float = 0.65,
    prefix_filter: bool = True
) -> pd.DataFrame:
    """
    Identifies semantically similar event labels using OpenAI embeddings.

    This function computes embeddings for all unique values in the
    'concept:name' column of the input DataFrame and calculates pairwise
    cosine similarity between them. Pairs with similarity above the given
    threshold are returned as potential synonym candidates.

    Optionally, labels can be filtered by prefix (e.g., 'W_', 'A_') to
    avoid comparing activities from different process sections.

    Args:
        df (pd.DataFrame): Input event log dataframe with a 'concept:name' column.
        threshold (float): Similarity threshold for considering two labels as synonyms.
        prefix_filter (bool): If True, only compare labels with the same prefix.

    Returns:
        pd.DataFrame: DataFrame containing pairs of similar labels and their similarity scores.
    """
    labels = df["concept:name"].unique()

    # --- embeddings ---
    embeddings = []
    for l in labels:
        res = openai_client.embeddings.create(
            model=DEPLOYMENT_NAME,
            input=l
        )
        embeddings.append(res.data[0].embedding)
    embeddings = np.array(embeddings)

    # --- similarity pairs ---
    results = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            if prefix_filter:
                if labels[i].split("_")[0] != labels[j].split("_")[0]:
                    continue
            sim = cosine_similarity(embeddings[i], embeddings[j])
            if sim > threshold:
                results.append({
                    "label_1": labels[i],
                    "label_2": labels[j],
                    "similarity": sim
                })
    return pd.DataFrame(results)

In [120]:
result = find_similar_labels(df_cleaned, threshold=0.65, prefix_filter = True)
result

,label_1,label_2,similarity
0,W_Complete application,W_Finalize application,0.667781
1,O_Sent (mail and online),O_Sent (online only),0.902907
2,W_Finish application,W_Finalize application,0.746618
3,W_Precheck application: Applicant Identity,W_Precheck application: Form Completeness,0.708304
4,W_Precheck application: Applicant Identity,W_Precheck application: Credit History,0.712661
5,W_Precheck application: Form Completeness,W_Precheck application: Credit History,0.663201


In [39]:
#Use domain knowledge to exclude label pairs with high similarity they don't make sense
SYNONYMOUS_LABELS = ["W_Complete application", "W_Finish application", "W_Finalize application"]
df_cleaned[df_cleaned["concept:name"].isin(SYNONYMOUS_LABELS)]["concept:name"].value_counts()

concept:name
W_Complete application    16875
W_Finalize application     3399
W_Finish application       3302
Name: count, dtype: Int64

In [40]:
def replace_with_most_frequent(
    df: pd.DataFrame,
    synonym_list: list,
    column: str = "concept:name"
) -> pd.DataFrame:
    """
    Replaces a group of synonymous event labels with the most frequent label.

    This function identifies the most common label within a given list of
    synonymous activities and replaces all occurrences of those labels in the
    DataFrame with the most frequent one.

    Args:
        df (pd.DataFrame): Input event log dataframe.
        synonym_list (list): List of synonymous labels to be replaced.
        column (str): Name of the column containing the labels (default is 'concept:name').
        
    Returns:
        pd.DataFrame: Copy of the dataframe with replaced labels.
    """
    df = df.copy()
    subset = df[df[column].isin(synonym_list)]
    most_frequent = subset[column].value_counts().idxmax()
    df.loc[df[column].isin(synonym_list), column] = most_frequent
    return df

In [41]:
df_cleaned = replace_with_most_frequent(df_cleaned, SYNONYMOUS_LABELS)
df_cleaned[df_cleaned["concept:name"].isin(SYNONYMOUS_LABELS)]["concept:name"].value_counts()

concept:name
W_Complete application    23576
Name: count, dtype: Int64

### Clean Colletaral Events
Coded by: Thien

In [42]:
def detect_collateral_events(
    df: pd.DataFrame,
    time_window: str = "5s",
    similarity: bool = False,
    similarity_df: pd.DataFrame = None,
    threshold: float = None
):
    """
    Detects potential collateral events based on temporal proximity
    and optional semantic similarity filtering.

    Args:
        df (pd.DataFrame): Input event log dataframe with 'case:concept:name', 'concept:name', and 'time:timestamp' columns.
        time_window (str): Maximum time difference between events to be considered collateral (e.g., '5s', '1m').
        similarity (bool): If True, apply semantic similarity filtering based on provided similarity_df.
        similarity_df (pd.DataFrame): DataFrame containing pairs of labels and their similarity scores (required if similarity=True).
        threshold (float): Minimum similarity score to consider labels related (required if similarity=True).
    
    Returns:
        pd.DataFrame: DataFrame containing pairs of potentially collateral events with their time difference and similarity score (if applicable).
    """

    df = df.copy()
    df = df.sort_values(["case:concept:name", "time:timestamp"])

    # --- build allowed similarity set ---
    allowed_pairs = set()
    if similarity and similarity_df is not None:
        sim_filtered = similarity_df[similarity_df["similarity"] >= threshold]
        for _, row in sim_filtered.iterrows():
            allowed_pairs.add((row["label_1"], row["label_2"]))
            allowed_pairs.add((row["label_2"], row["label_1"]))  # symmetric

    results = []
    for case_id, group in df.groupby("case:concept:name"):
        group = group.sort_values("time:timestamp")
        for i in range(len(group) - 1):
            row_a = group.iloc[i]
            row_b = group.iloc[i + 1]
            time_diff = row_b["time:timestamp"] - row_a["time:timestamp"]
            if time_diff <= pd.Timedelta(time_window):
                e1 = row_a["concept:name"]
                e2 = row_b["concept:name"]

                # --- semantic filter (optional) ---
                if similarity:
                    if (e1, e2) not in allowed_pairs:
                        continue
                results.append({
                    "case": case_id,
                    "event_1": e1,
                    "event_2": e2,
                    "time_diff": time_diff
                })

    return pd.DataFrame(results)

In [ ]:
result_for_collateral = result[~result['label_1'].isin(SYNONYMOUS_LABELS)]
collateral_df = detect_collateral_events(df_cleaned, time_window="5s", similarity=True, similarity_df=result_for_collateral, threshold=0.65)

In [129]:
set(collateral_df['event_1'].unique()) | set(collateral_df['event_2'].unique())

{'W_Precheck application: Applicant Identity',
 'W_Precheck application: Credit History',
 'W_Precheck application: Form Completeness'}

In [44]:
def merge_collateral_events(df, merge_dict, time_col="time:timestamp"):
    """
    Merges collateral sub-events into a single canonical event.

    For each group of events in merge_dict values:
    - All matching rows are replaced by the key label
    - Other attributes are taken from the earliest event in time

    Args:
        df (pd.DataFrame): Input event log dataframe.
        merge_dict (dict): Dictionary where keys are canonical labels and values are lists of sub-event labels to be merged.
        time_col (str): Name of the timestamp column to determine event order (default is 'time:timestamp').
    
    Returns:
        pd.DataFrame: DataFrame with collateral events merged into canonical events.
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    rows_to_drop = []
    rows_to_add = []

    for canonical, subevents in merge_dict.items():
        # process per case
        for case_id, group in df.groupby("case:concept:name"):
            subset = group[group["concept:name"].isin(subevents)]
            if subset.empty:
                continue
            # earliest event per case
            anchor = subset.sort_values(time_col).iloc[0]
            new_row = anchor.copy()
            new_row["concept:name"] = canonical
            rows_to_add.append(new_row)
            rows_to_drop.extend(subset.index.tolist())

    df = df.drop(index=rows_to_drop)
    df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)
    df = df.sort_values(["case:concept:name", time_col]).reset_index(drop=True)
    print("Rows to drop:", len(rows_to_drop))
    print("Rows added:", len(rows_to_add))
    return df

In [45]:
COLLATERAL_LABELS_DICT = {
  "W_Precheck application": [
    "W_Precheck application: Applicant Identity",
    "W_Precheck application: Form Completeness",
    "W_Precheck application: Credit History"
  ]
}
df_cleaned_final = merge_collateral_events(df_cleaned, COLLATERAL_LABELS_DICT)

Rows to drop: 9
Rows added: 3


## Data Quality Statistics
Coded by: Jonas

In [46]:
def print_stats(df):
    print(f"traces: {df['case:concept:name'].nunique()}")
    print(f"events: {len(df)}")
    print(f"distinct activities: {df['concept:name'].nunique()}")

    # ensure correct ordering for trace reconstruction
    df_sorted = df.copy()

    df_sorted["time:timestamp"] = pd.to_datetime(
        df_sorted["time:timestamp"],
        utc=True,
        errors="coerce"
    )

    df_sorted = df_sorted.sort_values(["case:concept:name", "time:timestamp"])

    # distinct traces / variants
    traces = []

    for _, group in df_sorted.groupby("case:concept:name"):
        trace = tuple(group["concept:name"].astype(str).tolist())
        traces.append(trace)

    distinct_traces = len(set(traces))

    # trace lengths
    trace_lengths = df_sorted.groupby("case:concept:name").size()

    print(f"distinct traces: {distinct_traces}")
    print(f"average trace length: {trace_lengths.mean():.2f}")
    print(f"min trace length: {trace_lengths.min()}")
    print(f"max trace length: {trace_lengths.max()}")

    # log duration
    log_duration = df_sorted["time:timestamp"].max() - df_sorted["time:timestamp"].min()
    print(f"log duration: {log_duration}")

    # missing values
    missing_values = df.isna().sum().sum()
    print(f"missing values: {missing_values}")

In [47]:
log_clean = pm4py.read_xes("../data/samples/clean_sample.xes.gz")
df_clean = pm4py.convert_to_dataframe(log_clean)

parsing log, completed traces :: 100%|██████████| 5000/5000 [00:13<00:00, 374.55it/s]


In [48]:
print("Statistics for clean log")
print_stats(df_clean)

print("\nStatistics for noised log")
print_stats(df_noised)

print("\nStatistics for recoverd log")
print_stats(df_cleaned_final)

Statistics for clean log
traces: 5000
events: 190334
distinct activities: 25
distinct traces: 3132
average trace length: 38.07
min trace length: 10
max trace length: 180
log duration: 395 days 21:09:23.037000
missing values: 1450497

Statistics for noised log
traces: 5000
events: 151673
distinct activities: 202
distinct traces: 4745
average trace length: 30.33
min trace length: 10
max trace length: 146
log duration: 395 days 21:09:23.034000
missing values: 1285299

Statistics for recoverd log
traces: 5000
events: 190337
distinct activities: 26
distinct traces: 3207
average trace length: 38.07
min trace length: 10
max trace length: 180
log duration: 395 days 21:09:23.214000
missing values: 1862092


## Export Recovered XES

In [49]:
event_log_recovered = pm4py.convert_to_event_log(df_cleaned_final)

In [50]:
pm4py.write_xes(event_log_recovered, "../data/samples/recovered_sample.xes.gz")

exporting log, completed traces :: 100%|██████████| 5000/5000 [00:12<00:00, 408.46it/s]
